# Overhead comparison: JANUS-FL's real crypto pipeline vs. plain FedAvg

Closes brutal-review gap #3: "no quantified overhead vs. plain FedAvg."
Same model (`tiny_cnn`, ~20.6K params), same data (MNIST, case1 IID
split), same 50 clients, same per-client local-training step (both
loops call the real, unmodified `fl/client.py::local_train_delta`) --
the ONLY thing that differs is the aggregation layer: plain mean (zero
crypto) vs. JANUS-FL's real masked-and-verified secure aggregate
(`fl/round_loop.py::run_training_loop`, completely unmodified).

**Two measurements, both real, neither estimated:**
1. Per-round wall-clock, timed with `time.perf_counter()` around each
   loop's real call, 3 rounds each (enough for a stable average; not
   an accuracy claim -- the already-confirmed 20-round accuracy
   numbers in STATUS.md's TWENTY-FOURTH item are unaffected by and
   unrelated to this notebook).
2. Per-client communication size: one real client's actual pickled
   `ClientSubmissionPhase1` + `ClientSubmissionPhase2` objects vs. the
   same client's raw trained delta, pickled the same way for a fair
   byte-for-byte comparison.

**Zero risk to the existing pipeline**: this notebook only ever calls
already-existing, unmodified functions (`local_train_delta`,
`run_training_loop`, `round_step_phase1`/`round_step_phase2`) plus a
brand-new, standalone `baselines/plain_fedavg_loop.py` that has no
coupling back into `fl/`, `crypto/`, or `attacks/` -- nothing here can
affect the real pipeline's own already-confirmed results.


In [ ]:
# 1. Clone/pull the repo. (No Drive mount -- checkpoints go through
# git instead, see the git-checkpoint-setup cell further down and this
# notebook's own intro markdown for why.)
import os

if not os.path.exists('/content/FL-ZKP'):
    !git clone https://github.com/YashasweeNarayan/FL-ZKP.git /content/FL-ZKP
else:
    !cd /content/FL-ZKP && git fetch -q origin main && git reset -q --hard origin/main
%cd /content/FL-ZKP


In [ ]:
# 2. Install the EC-dependent toolchain (rbcl for ristretto255, torch/torchvision, flwr-datasets).
!pip install -q rbcl
!pip install -q pynacl
!pip install -q flwr-datasets[vision]
!pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu121


**If this is the first time cell 2 has installed anything in this
session, restart the kernel now: Runtime -> Restart session.** Then
continue from cell 3 below (NOT from cell 1 -- Drive stays mounted and
the repo stays cloned across a restart, only the Python process
resets, so re-running cell 1 would be harmless but pointless).


In [ ]:
# 3. Sanity-check the install.
import sys
sys.path.insert(0, '/content/FL-ZKP')
import rbcl
assert hasattr(rbcl, "crypto_core_ristretto255_from_hash"), (
    "rbcl imported but is missing crypto_core_ristretto255_from_hash -- "
    "see crypto/ristretto.py's import fallback and FIX.md F8."
)
import torch
print("rbcl OK. torch OK, CUDA available:", torch.cuda.is_available())


In [ ]:
# 4. Load config.
import yaml
with open('configs/mvp_3day.yaml') as f:
    CFG = yaml.safe_load(f)
print(CFG)


## Data + model (shared by both loops)

In [ ]:
# 5. Load MNIST, stratified-IID split across clients.
# NOTE: configs/mvp_3day.yaml's datasets.mnist.source is "csv" (reads a
# Drive-cached CSV that this run's Drive doesn't have) -- that path is
# for whoever already staged the CSV. This run instead forces
# source="torchvision", which downloads MNIST directly over the network,
# no Drive dependency at all. Deliberately NOT changed in the shared
# config file, since other runs may rely on the csv path being default.
from fl.data_iid import prepare_case1

case1_data = prepare_case1(seed=CFG['seed'], source="torchvision")
print("clients:", len(case1_data['client_train']))
print("server_root size:", len(case1_data['server_root'].labels))
print("global_val size:", len(case1_data['global_val'].labels))


In [ ]:
# 6. Build the ring (one keypair per client) and the trace committee.
from crypto.ristretto import Scalar, scalar_mul_base
from crypto.dkg import generate_contribution, verify_sub_share, finalize, pk_share_for
from fl.round_loop import build_x25519_keys, precompute_pairwise_secrets

n_clients = CFG['protocol']['n_clients']
t = CFG['protocol']['threshold']
n_c = CFG['protocol']['n_key_holders']

client_sks = [Scalar.random() for _ in range(n_clients)]
ring = [scalar_mul_base(sk) for sk in client_sks]

x25519_sks, x25519_pks = build_x25519_keys(n_clients)
pairwise_secrets = precompute_pairwise_secrets(x25519_sks, x25519_pks)

key_holder_ids = list(range(1, n_c + 1))
dkg_contributions = [generate_contribution(h, t, key_holder_ids) for h in key_holder_ids]

# Feldman-verify every sub-share before finalizing -- a real committee
# would discard a contribution that fails this and re-run DKG without
# the cheating holder; all contributions here are honest, so every
# check is expected to pass. Same check notebooks/traceability_demo
# already runs standalone; added here (2026-11-19) alongside pk_shares
# below so this notebook's committee is held to the same standard, now
# that §6.2 opening runs for real from this notebook's own evidence
# (not just the standalone demo).
for c in dkg_contributions:
    for j in key_holder_ids:
        assert verify_sub_share(c.sub_shares[j], c.feldman_commitments), (
            f"holder {c.holder_id}'s sub-share to {j} failed Feldman verification"
        )

PK_trace, key_holder_shares = finalize(dkg_contributions, key_holder_ids)
# pk_shares (2026-11-19): each holder's PUBLIC Feldman commitment --
# needed by crypto.elgamal_threshold.verify_partial inside
# fl/trace.py::run_opening_protocol to check a partial decryption
# before combining it. key_holder_shares (above) are the holders' own
# SECRET shares, needed to CREATE a partial in the first place
# (crypto.elgamal_threshold.compute_partial) -- both come out of the
# SAME DKG run, never recomputed independently.
pk_shares = {i: pk_share_for(i, dkg_contributions) for i in key_holder_ids}
print(f"ring of {len(ring)} published, PK_trace established among {n_c} key holders (t={t})")
print(f"{n_clients} X25519 keypairs + {n_clients*(n_clients-1)} pairwise secrets precomputed")


In [ ]:
# 6.5. §6.2 real opening-protocol handler, wired to
# run_training_loop's on_pseudonym_flagged callback (fl/round_loop.py,
# 2026-11-19). This is the SAME run_opening_protocol +
# revoke + residual_anonymity_bits machinery already validated
# standalone in notebooks/traceability_demo/TRACEABILITY_DEMO.ipynb
# (both against the local pure-Python backend and, per FIX.md, the
# real rbcl backend on Colab) -- nothing new is introduced here, this
# cell just triggers it from REAL per-round evidence instead of a
# hand-built test case, using the SAME committee (key_holder_shares,
# pk_shares) the ring cell above just built for escrow.
#
# None of these key holders is ever one of this run's clients (two
# disjoint simulated roles), so no holder ever recuses as
# the accused party (fl/trace.py::collect_partial_from_holder's
# recusal rule) -- every holder contributes a partial to every opening
# attempt, same as the demo notebook's partials_from() helper.
from crypto.elgamal_threshold import compute_partial
from fl.trace import OpeningRequest, run_opening_protocol, revoke, residual_anonymity_bits

traceability_log = []       # one dict per flagged pseudonym: round, opened, H(T), etc.
_opening_ring = list(ring)  # REPORTING-ONLY shrinking ring, for the H(T) curve -- does NOT
                             # feed back into this run's own ring-signature verification (that
                             # would need §6.4 committee/ring reconfiguration, out of scope here;
                             # see FIX.md's traceability-wiring entries for why).
_n_opened_so_far = 0


def on_pseudonym_flagged(pseudonym_bytes, t_round, evidence_total, escrow_ciphertext, epoch, certificate):
    global _opening_ring, _n_opened_so_far
    request = OpeningRequest(
        epoch=epoch, accused_pseudonym=pseudonym_bytes, evidence_total=evidence_total,
        calibration_certificate=certificate, escrow_ciphertext=escrow_ciphertext,
    )
    transcript = f"opening:{epoch}:{pseudonym_bytes.hex()}".encode()
    partials = {
        i: compute_partial(escrow_ciphertext, key_holder_shares[i], pk_shares[i], transcript)
        for i in key_holder_ids
    }
    verdict = run_opening_protocol(request, partials, pk_shares, t, admin_authorized=True)

    entry = {
        "round": t_round, "epoch": epoch, "pseudonym": pseudonym_bytes.hex(),
        "evidence_total": evidence_total, "theta": certificate.theta, "opened": verdict.opened,
    }
    if verdict.opened:
        _opening_ring, record = revoke(verdict, _opening_ring, epoch)
        _n_opened_so_far += 1
        entry["new_ring_size"] = record.new_ring_size
        entry["residual_anonymity_bits"] = residual_anonymity_bits(len(ring), _n_opened_so_far)
        print(f"  round {t_round}: TRACEABILITY -- pseudonym {pseudonym_bytes.hex()[:12]}... OPENED "
              f"(evidence={evidence_total:.3f} > theta={certificate.theta:.3f}), "
              f"ring {len(ring)}->{record.new_ring_size}, H(T)={entry['residual_anonymity_bits']:.3f} bits")
    else:
        entry["reason"] = verdict.reason
        print(f"  round {t_round}: TRACEABILITY -- pseudonym {pseudonym_bytes.hex()[:12]}... flagged but "
              f"NOT opened ({verdict.reason})")
    traceability_log.append(entry)


In [ ]:
# 7. Build the model.
# ~20.6K params, matches WhiteCloak/PQS-BFL's model scale.
import torch
from fl.models import build_model, flatten_state_dict

REAL_MODEL = 'tiny_cnn'
model = build_model(REAL_MODEL, num_classes=10, in_channels=1)

# GPU fix (2026-11-16): build_model() returns a plain nn.Module on
# CPU -- nothing downstream (fl/client.py, fl/round_loop.py) ever
# moved it or the per-batch data onto CUDA before this fix, so training
# ran entirely on CPU even on a GPU Colab runtime. Moving the model
# here, once, before the run starts is now the ONLY place that needs
# to pick the device -- fl/client.py's local_train_delta reads
# `next(model.parameters()).device` and moves each mini-batch to match,
# and fl/round_loop.py's checkpoint/resume path follows the same model
# device automatically (see both files' own 2026-11-16 comments).
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)
print("training device:", device)

flat_w, model_shapes = flatten_state_dict(model)
dim = flat_w.numel()
print(REAL_MODEL, "param count:", dim)


## JANUS-FL's real crypto pipeline -- 3 rounds, timed

In [ ]:
import time
from fl.round_loop import run_training_loop, bootstrap_norms_from_model_dim, release_gpu_memory

# Reuses `model`/`dim`/`model_shapes` the model cell above already
# built (REAL_MODEL='tiny_cnn'), and the `ring`/`client_sks`/
# `pairwise_secrets`/`PK_trace` the ring + opening-handler cells above
# already built for this notebook's n_clients clients -- nothing
# rebuilt, so this is the exact same committee/ring machinery every
# other MVP run notebook exercises.
crypto_root_norms = bootstrap_norms_from_model_dim(
    model_dim=dim, sketch_dim=CFG['sketch']['m'],
    sketch_seed=b"bootstrap-overhead-comparison", quant_bits=16,
    rng=np.random.default_rng(CFG['seed'] + 3),
)

_t0 = time.perf_counter()
_, crypto_history = run_training_loop(
    model=model, model_dim=dim,
    client_datasets=case1_data['client_train'][:n_clients],
    ring=ring, client_sks=client_sks,
    pairwise_secrets=pairwise_secrets,
    pk_trace=PK_trace, server_root_norms=crypto_root_norms,
    CFG=CFG, rounds=3,
    on_pseudonym_flagged=on_pseudonym_flagged,
)
_total_crypto_time = time.perf_counter() - _t0
release_gpu_memory()

print(f"JANUS-FL total wall-clock for 3 rounds (s): {_total_crypto_time:.2f}")
print(f"JANUS-FL mean per-round time: {_total_crypto_time/3:.2f}s")
print("per-round accepted counts (sanity, not an accuracy claim):",
      [h.get('accepted') for h in crypto_history if not h.get('aborted')])


## Per-client communication size -- one real submission, measured not estimated

In [ ]:
import pickle
from fl.client import ClientRoundContext, ClientKeys, round_step_phase1, round_step_phase2
from crypto.secagg import new_self_mask_secret

# One real client's submission, built the same way round 0 of the real
# protocol would build it -- client index 0 of the SAME ring/committee
# just used above, same (now-trained) model, same local data.
comm_ctx = ClientRoundContext(
    keys=ClientKeys(sk=client_sks[0], pk=ring[0], index=0),
    ring=ring,
    pairwise_secrets=pairwise_secrets[0],
    self_mask_secret_int=new_self_mask_secret(),
    blinding_self_mask_secret_int=new_self_mask_secret(),
    model_dim=dim,
)
comm_w_t, _ = flatten_state_dict(model)
comm_w_t = comm_w_t.detach().cpu().numpy().astype(np.float64)

comm_phase1 = round_step_phase1(
    comm_ctx, comm_w_t, rho_t=b"comm-size-rho-t", rho_e=b"comm-size-rho-e",
    vrf_alpha=b"comm-size-vrf-alpha", sampling_rate=1.0, sigma_dp=0.0,
    n_clients=n_clients, local_data=case1_data['client_train'][0], model=model,
    lr=CFG['local_training']['lr'], epochs=CFG['local_training']['epochs'],
    batch_size=CFG['local_training']['batch_size'], pk_trace=PK_trace,
)
comm_phase2 = round_step_phase2(
    comm_ctx, comm_phase1, sketch_seed=CFG['sketch'].get('seed', b'sigma-t-comm-size'),
    sketch_dim=CFG['sketch']['m'], B_t=1e12, rho_t=b"comm-size-rho-t",
    rho_e=b"comm-size-rho-e", vrf_alpha=b"comm-size-vrf-alpha",
)

raw_delta = np.random.default_rng(0).normal(0, 1, size=dim).astype(np.float64)
raw_delta_bytes = len(pickle.dumps(raw_delta))
submission_bytes = len(pickle.dumps(comm_phase1)) + len(pickle.dumps(comm_phase2))

print(f"model_dim = {dim}")
print(f"raw delta, pickled: {raw_delta_bytes:,} bytes")
print(f"JANUS-FL submission (phase1 + phase2), pickled: {submission_bytes:,} bytes")
print(f"overhead factor: {submission_bytes / raw_delta_bytes:.2f}x")


## Plain FedAvg -- same 3 rounds, same scale, timed, zero crypto

In [ ]:
import time
import sys
sys.path.insert(0, '/content/FL-ZKP')
from baselines.plain_fedavg_loop import run_plain_fedavg_loop
from baselines.aggregation_rules import mean_aggregate

# A SEPARATE, freshly-initialized model -- the crypto loop above has
# already trained `model` in place, so this loop gets its own
# independent copy rather than continuing from trained weights, for a
# fair round-0-to-round-0 timing comparison.
plain_model = build_model('tiny_cnn', num_classes=10, in_channels=1).to(device)
plain_flat, plain_shapes = flatten_state_dict(plain_model)
plain_dim = plain_flat.numel()

plain_result = run_plain_fedavg_loop(
    model=plain_model,
    model_dim=plain_dim,
    client_datasets=case1_data['client_train'][:n_clients],
    rounds=3,
    lr=CFG['local_training']['lr'],
    epochs=CFG['local_training']['epochs'],
    batch_size=CFG['local_training']['batch_size'],
    aggregator=mean_aggregate,
    seed=CFG['seed'],
)
plain_round_times = [r.wall_clock_seconds for r in plain_result.history]
print("plain FedAvg per-round wall-clock (s):", [f"{t:.2f}" for t in plain_round_times])
print(f"plain FedAvg mean per-round time: {sum(plain_round_times)/len(plain_round_times):.2f}s")


## Summary

In [ ]:
print("\n" + "="*72)
print("OVERHEAD COMPARISON SUMMARY -- plain FedAvg vs. JANUS-FL")
print("="*72)
print(f"model: tiny_cnn (~20.6K params), 50 clients, {plain_dim} model_dim")
print()
_plain_mean = sum(plain_round_times) / len(plain_round_times)
_crypto_mean = _total_crypto_time / 3
print(f"plain FedAvg:  {_plain_mean:.2f}s/round")
print(f"JANUS-FL:      {_crypto_mean:.2f}s/round")
print(f"time overhead: {_crypto_mean / _plain_mean:.1f}x")
print()
print(f"raw delta size:        {raw_delta_bytes:,} bytes")
print(f"JANUS-FL submission:   {submission_bytes:,} bytes")
print(f"comm. overhead:        {submission_bytes / raw_delta_bytes:.1f}x")
print()
print("NOTE: this is a timing/size measurement at 3 rounds for a stable")
print("average, NOT a replacement for the already-confirmed 20-round accuracy")
print("numbers (STATUS.md TWENTY-FOURTH item) -- those are unaffected.")
